# GOLD: summary tables for reporting

Builds three small tables from the clean Silver data. These are the tables a sales dashboard reads:

1. Revenue by month, with growth from the month before.
2. Sales by product, ranked by revenue and by units sold.
3. Sales by country, with each country's share of revenue.

## Table 1: revenue by month

`LAG` looks up the revenue of the month before, so growth can be calculated. December 2011 only has 9 days of data, so it is flagged as a partial month.

In [ ]:
%sql
create or replace table procat.retail.gold_monthly_revenue as
with monthly as (
    select
        cast(date_trunc('MONTH', invoice_date) as date) as order_month,
        sum(line_amount) as revenue,
        count(distinct invoice) as orders,
        count(distinct customer_id) as customers
    from procat.retail.silver_order_lines
    group by cast(date_trunc('MONTH', invoice_date) as date)
)
select
    order_month,
    round(revenue, 2) as revenue,
    orders,
    customers,
    round(revenue / orders, 2) as avg_order_value,
    round((revenue - lag(revenue) over (order by order_month)) / lag(revenue) over (order by order_month) * 100, 1) as growth_pct,
    order_month = date '2011-12-01' as is_partial_month
from monthly;

In [ ]:
%sql
select * from procat.retail.gold_monthly_revenue order by order_month;

## Table 2: sales by product

Postage and carriage codes (POST, DOT, C2) are left out, because they are fees and not products. `RANK` gives each product its position by revenue and by units sold.

In [ ]:
%sql
create or replace table procat.retail.gold_product_sales as
with products as (
    select
        stock_code,
        description,
        sum(line_amount) as revenue,
        sum(quantity) as units_sold
    from procat.retail.silver_order_lines
    where stock_code not in ('POST', 'DOT', 'C2')
    group by stock_code, description
)
select
    stock_code,
    description,
    round(revenue, 2) as revenue,
    units_sold,
    rank() over (order by revenue desc) as revenue_rank,
    rank() over (order by units_sold desc) as units_rank
from products;

In [ ]:
%sql
select * from procat.retail.gold_product_sales where revenue_rank <= 10 order by revenue_rank;

## Table 3: sales by country

`SUM(...) OVER ()` gives the total across all countries, so each country's share can be calculated in the same query.

In [ ]:
%sql
create or replace table procat.retail.gold_country_sales as
select
    country,
    round(sum(line_amount), 2) as revenue,
    count(distinct customer_id) as customers,
    count(distinct invoice) as orders,
    round(sum(line_amount) / count(distinct invoice), 2) as avg_order_value,
    round(sum(line_amount) / sum(sum(line_amount)) over () * 100, 2) as pct_of_revenue
from procat.retail.silver_order_lines
group by country;

In [ ]:
%sql
select * from procat.retail.gold_country_sales order by revenue desc limit 10;

## Final check: the whole pipeline in one table

In [ ]:
%sql
select 'bronze_order_lines' as table_name, count(*) as row_count from procat.retail.bronze_order_lines
union all
select 'silver_order_lines', count(*) from procat.retail.silver_order_lines
union all
select 'silver_rejected_lines', count(*) from procat.retail.silver_rejected_lines
union all
select 'gold_monthly_revenue', count(*) from procat.retail.gold_monthly_revenue
union all
select 'gold_product_sales', count(*) from procat.retail.gold_product_sales
union all
select 'gold_country_sales', count(*) from procat.retail.gold_country_sales;